In [0]:

from pyspark.sql import functions as F

path_a = "/Volumes/workspace/finance/raw_files/bank_a"

# Our own column names, matched BY POSITION to the 6 columns in the file
schema_a = """
    date_raw STRING,
    narration STRING,
    ref_no STRING,
    withdrawal_raw STRING,
    deposit_raw STRING,
    balance_raw STRING
"""

df_a_raw = (spark.read.format("csv")
            .option("header", "true")      # skip the bank's header row
            .schema(schema_a)              # use our names and types instead
            .load(path_a))

df_a_raw.printSchema()
display(df_a_raw)

root
 |-- date_raw: string (nullable = true)
 |-- narration: string (nullable = true)
 |-- ref_no: string (nullable = true)
 |-- withdrawal_raw: string (nullable = true)
 |-- deposit_raw: string (nullable = true)
 |-- balance_raw: string (nullable = true)



date_raw,narration,ref_no,withdrawal_raw,deposit_raw,balance_raw
01/07/26,NEFT-N6941266115-ACME TECHNOLOGIES PVT LTD-SALARY JUL,742669170937,null,"95,000.00","325,268.89"
02/07/26,UPI/ZOMATO/zomato@hdfcbank/Order/427928307,884039220820,576.74,null,"324,692.15"
03/07/26,UPI/BIGBASKET/bigbasket@icici/Groceries/700155683,518008488562,"1,519.82",null,"323,172.33"
05/07/26,IMPS-557267663034-RAJESH KUMAR-HOUSE RENT,757106922666,"18,000.00",null,"305,172.33"
05/07/26,UPI/SWIGGY/swiggy@axisbank/Food order/634447845,424401271462,193.64,null,"304,978.69"
06/07/26,UPI/UBER INDIA/uber@axis/Trip/545809626,655999474067,380.71,null,"304,597.98"
07/07/26,UPI/GREEN LEAF VEGETABLES/paytmqr@paytm/Payment/128063847,575046690837,321.15,null,"304,276.83"
08/07/26,UPI/BLINKIT/blinkit@ybl/Quick order/190310715,972304329325,439.24,null,"303,837.59"
10/07/26,ACH/ELECTRICITY BOARD/32360572,227568171232,"2,399.47",null,"301,438.12"
10/07/26,UPI/SWIGGY/swiggy@axisbank/Food order/517729190,820922255341,333.86,null,"301,104.26"


In [0]:
df_a_step2 = df_a_raw.withColumn(
    "txn_date",
    F.expr("to_date(try_to_timestamp(trim(date_raw), 'dd/MM/yy'))")
)

display(df_a_step2.select("date_raw", "txn_date"))

date_raw,txn_date
01/07/26,2026-07-01
02/07/26,2026-07-02
03/07/26,2026-07-03
05/07/26,2026-07-05
05/07/26,2026-07-05
06/07/26,2026-07-06
07/07/26,2026-07-07
08/07/26,2026-07-08
10/07/26,2026-07-10
10/07/26,2026-07-10


In [0]:
def to_amount(col_name):
    """'72,000.00' -> 72000.00 ; '' or garbage -> NULL instead of an error."""
    return F.expr(f"try_cast(regexp_replace(trim({col_name}), ',', '') AS DECIMAL(12,2))")

df_a_step3 = (df_a_step2
              .withColumn("withdrawal", to_amount("withdrawal_raw"))
              .withColumn("deposit", to_amount("deposit_raw")))

display(df_a_step3.select("withdrawal_raw", "withdrawal", "deposit_raw", "deposit"))

withdrawal_raw,withdrawal,deposit_raw,deposit
null,null,"95,000.00",95000.00
576.74,576.74,null,null
"1,519.82",1519.82,null,null
"18,000.00",18000.00,null,null
193.64,193.64,null,null
380.71,380.71,null,null
321.15,321.15,null,null
439.24,439.24,null,null
"2,399.47",2399.47,null,null
333.86,333.86,null,null


In [0]:
df_a_step4 = (df_a_step3
    .withColumn("amount",
                F.when(F.col("withdrawal") > 0, F.col("withdrawal"))
                 .otherwise(F.col("deposit")))
    .withColumn("direction",
                F.when(F.col("withdrawal") > 0, F.lit("DEBIT"))
                 .when(F.col("deposit") > 0, F.lit("CREDIT"))))   # neither -> NULL

display(df_a_step4.select("withdrawal", "deposit", "amount", "direction"))

withdrawal,deposit,amount,direction
null,95000.00,95000.00,CREDIT
576.74,null,576.74,DEBIT
1519.82,null,1519.82,DEBIT
18000.00,null,18000.00,DEBIT
193.64,null,193.64,DEBIT
380.71,null,380.71,DEBIT
321.15,null,321.15,DEBIT
439.24,null,439.24,DEBIT
2399.47,null,2399.47,DEBIT
333.86,null,333.86,DEBIT


In [0]:
df_a_clean = (df_a_step4
    .withColumn("source_file", F.col("_metadata.file_name"))
    .select(
        "txn_date",
        F.trim("narration").alias("description"),
        F.trim("ref_no").alias("bank_ref"),
        "amount",
        "direction",
        "source_file",
    ))

df_a_clean.printSchema()
display(df_a_clean)

root
 |-- txn_date: date (nullable = true)
 |-- description: string (nullable = true)
 |-- bank_ref: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- direction: string (nullable = true)
 |-- source_file: string (nullable = false)



txn_date,description,bank_ref,amount,direction,source_file
2026-07-01,NEFT-N6941266115-ACME TECHNOLOGIES PVT LTD-SALARY JUL,742669170937,95000.00,CREDIT,bank_a_statement_2026_07.csv
2026-07-02,UPI/ZOMATO/zomato@hdfcbank/Order/427928307,884039220820,576.74,DEBIT,bank_a_statement_2026_07.csv
2026-07-03,UPI/BIGBASKET/bigbasket@icici/Groceries/700155683,518008488562,1519.82,DEBIT,bank_a_statement_2026_07.csv
2026-07-05,IMPS-557267663034-RAJESH KUMAR-HOUSE RENT,757106922666,18000.00,DEBIT,bank_a_statement_2026_07.csv
2026-07-05,UPI/SWIGGY/swiggy@axisbank/Food order/634447845,424401271462,193.64,DEBIT,bank_a_statement_2026_07.csv
2026-07-06,UPI/UBER INDIA/uber@axis/Trip/545809626,655999474067,380.71,DEBIT,bank_a_statement_2026_07.csv
2026-07-07,UPI/GREEN LEAF VEGETABLES/paytmqr@paytm/Payment/128063847,575046690837,321.15,DEBIT,bank_a_statement_2026_07.csv
2026-07-08,UPI/BLINKIT/blinkit@ybl/Quick order/190310715,972304329325,439.24,DEBIT,bank_a_statement_2026_07.csv
2026-07-10,ACH/ELECTRICITY BOARD/32360572,227568171232,2399.47,DEBIT,bank_a_statement_2026_07.csv
2026-07-10,UPI/SWIGGY/swiggy@axisbank/Food order/517729190,820922255341,333.86,DEBIT,bank_a_statement_2026_07.csv


In [0]:
# Rows that didn't convert cleanly
display(df_a_clean.filter("txn_date IS NULL OR amount IS NULL OR direction IS NULL"))

# Quick totals per direction
display(df_a_clean.groupBy("direction").agg(
    F.count("*").alias("rows"),
    F.sum("amount").alias("total")))

txn_date,description,bank_ref,amount,direction,source_file
null,UPI/SWIGGY/swiggy@axisbank/Food order/352130348,716328933739,250.00,DEBIT,bank_a_statement_2026_07.csv


direction,rows,total
DEBIT,194,394994.36
CREDIT,11,1045000.00


In [0]:
print(df_a_clean.columns)

['txn_date', 'description', 'bank_ref', 'amount', 'direction', 'source_file']


In [0]:
from pyspark.sql import functions as F

path_b = "/Volumes/workspace/finance/raw_files/bank_b"

# DIFFERENT: only 4 columns
schema_b = """
    date_raw STRING,
    description_raw STRING,
    amount_raw STRING,
    type_raw STRING
"""

df_b_raw = (spark.read.format("csv")
            .option("header", "true")
            .schema(schema_b)
            .load(path_b))

df_b_raw.printSchema()
display(df_b_raw)


root
 |-- date_raw: string (nullable = true)
 |-- description_raw: string (nullable = true)
 |-- amount_raw: string (nullable = true)
 |-- type_raw: string (nullable = true)



date_raw,description_raw,amount_raw,type_raw
2026-05-01,POS 453212******1234 STARBUCKS COFFEE,395.00,DR
2026-05-03,NETFLIX.COM SUBSCRIPTION,649.00,DR
2026-05-03,POS 453212******1234 AMAZON PAY IN,"1,148.64",DR
2026-05-07,SPOTIFY INDIA,119.00,DR
2026-05-08,DINEOUT*BARBEQUE NATION,"2,696.02",DR
2026-05-12,DINEOUT*BARBEQUE NATION,"2,008.94",DR
2026-05-15,POS 453212******1234 AMAZON PAY IN,642.52,DR
2026-05-16,PAYMENT RECEIVED - THANK YOU,"11,658.11",CR
2026-05-17,POS 453212******1234 MYNTRA DESIGNS,"2,322.27",DR
2026-05-18,POS 453212******1234 STARBUCKS COFFEE,345.00,DR


In [0]:
def to_amount(col_name):
    return F.expr(f"try_cast(regexp_replace(trim({col_name}), ',', '') AS DECIMAL(12,2))")

df_b_step2 = (df_b_raw
    # DIFFERENT: ISO date format
    .withColumn("txn_date", F.expr("to_date(try_to_timestamp(trim(date_raw), 'yyyy-MM-dd'))"))
    # DIFFERENT: one amount column. abs() guarantees it's positive
    .withColumn("amount", F.abs(to_amount("amount_raw"))))

display(df_b_step2.select("date_raw", "txn_date", "amount_raw", "amount"))

date_raw,txn_date,amount_raw,amount
2026-05-01,2026-05-01,395.00,395.00
2026-05-03,2026-05-03,649.00,649.00
2026-05-03,2026-05-03,"1,148.64",1148.64
2026-05-07,2026-05-07,119.00,119.00
2026-05-08,2026-05-08,"2,696.02",2696.02
2026-05-12,2026-05-12,"2,008.94",2008.94
2026-05-15,2026-05-15,642.52,642.52
2026-05-16,2026-05-16,"11,658.11",11658.11
2026-05-17,2026-05-17,"2,322.27",2322.27
2026-05-18,2026-05-18,345.00,345.00


In [0]:
df_b_step3 = df_b_step2.withColumn(
    "direction",
    # DIFFERENT: direction comes from DR/CR, not from which column is filled
    F.when(F.upper(F.trim("type_raw")).isin("DR", "DEBIT"), F.lit("DEBIT"))
     .when(F.upper(F.trim("type_raw")).isin("CR", "CREDIT"), F.lit("CREDIT"))
)

display(df_b_step3.select("type_raw", "direction"))

type_raw,direction
DR,DEBIT
DR,DEBIT
DR,DEBIT
DR,DEBIT
DR,DEBIT
DR,DEBIT
DR,DEBIT
CR,CREDIT
DR,DEBIT
DR,DEBIT


In [0]:
df_b_clean = (df_b_step3
    .withColumn("source_file", F.col("_metadata.file_name"))
    .select(
        "txn_date",
        F.trim("description_raw").alias("description"),
        F.lit(None).cast("string").alias("bank_ref"),   # DIFFERENT: card statements have no ref number
        "amount",
        "direction",
        "source_file",
    ))

df_b_clean.printSchema()
display(df_b_clean)

root
 |-- txn_date: date (nullable = true)
 |-- description: string (nullable = true)
 |-- bank_ref: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- direction: string (nullable = true)
 |-- source_file: string (nullable = false)



txn_date,description,bank_ref,amount,direction,source_file
2026-05-01,POS 453212******1234 STARBUCKS COFFEE,null,395.00,DEBIT,bank_b_card_2026_05.csv
2026-05-03,NETFLIX.COM SUBSCRIPTION,null,649.00,DEBIT,bank_b_card_2026_05.csv
2026-05-03,POS 453212******1234 AMAZON PAY IN,null,1148.64,DEBIT,bank_b_card_2026_05.csv
2026-05-07,SPOTIFY INDIA,null,119.00,DEBIT,bank_b_card_2026_05.csv
2026-05-08,DINEOUT*BARBEQUE NATION,null,2696.02,DEBIT,bank_b_card_2026_05.csv
2026-05-12,DINEOUT*BARBEQUE NATION,null,2008.94,DEBIT,bank_b_card_2026_05.csv
2026-05-15,POS 453212******1234 AMAZON PAY IN,null,642.52,DEBIT,bank_b_card_2026_05.csv
2026-05-16,PAYMENT RECEIVED - THANK YOU,null,11658.11,CREDIT,bank_b_card_2026_05.csv
2026-05-17,POS 453212******1234 MYNTRA DESIGNS,null,2322.27,DEBIT,bank_b_card_2026_05.csv
2026-05-18,POS 453212******1234 STARBUCKS COFFEE,null,345.00,DEBIT,bank_b_card_2026_05.csv


In [0]:
display(df_b_clean.filter("txn_date IS NULL OR amount IS NULL OR direction IS NULL"))

txn_date,description,bank_ref,amount,direction,source_file
2026-05-15,POS 453212******1234 AMAZON PAY IN,null,null,DEBIT,bank_b_card_2026_05.csv


In [0]:
print(df_a_clean.columns)
print(df_b_clean.columns)
print(df_a_clean.schema == df_b_clean.schema)

['txn_date', 'description', 'bank_ref', 'amount', 'direction', 'source_file']
['txn_date', 'description', 'bank_ref', 'amount', 'direction', 'source_file']
True


In [0]:
df_a_clean.write.mode("overwrite").saveAsTable("workspace.finance.bank_a_clean")
df_b_clean.write.mode("overwrite").saveAsTable("workspace.finance.bank_b_clean")
print("saved")

saved
